## LangGraph Agent를 위한 AgentCore Evaluations 온라인 평가

이 튜토리얼에서는 LangGraph 에이전트에 AgentCore Evaluations의 온라인 평가를 적용하는 방법을 알아봅니다.

이 실습을 진행하려면 먼저 [00-prereqs](../../00-prereqs) 폴더의 코드로 LangGraph 에이전트를 생성하고, [01-creating-custom-evaluators](../../01-creating-custom-evaluators)의 코드로 사용자 지정 evaluator를 생성해야 합니다.

### 학습 내용
- AgentCore Starter toolkit을 사용하여 trace에 온라인 평가를 실행하는 방법

### 튜토리얼 세부 정보

| 정보                | 세부 정보                                                                     |
|:--------------------|:------------------------------------------------------------------------------|
| 튜토리얼 유형       | 온라인 evaluator(기본 제공 및 사용자 지정)를 사용한 LangGraph 에이전트 평가 |
| 튜토리얼 구성 요소 | 기본 제공 및 사용자 지정 evaluator를 사용한 자동 평가 설정                   |
| 튜토리얼 분야       | 산업군 공통                                                                    |
| 예제 난이도         | 쉬움                                                                           |
| 사용한 SDK          | Amazon Bedrock AgentCore Starter toolkit                                       |

### 온라인 평가

온라인 평가는 배포된 에이전트의 실시간 트래픽 품질을 모니터링할 수 있게 해 줍니다. 선택한 특정 상호 작용을 분석하는 온디맨드 평가와 달리, 온라인 평가는 실시간 트래픽을 기반으로 프로덕션 환경의 에이전트 성능을 지속적으로 평가합니다.

온라인 평가는 세 가지 주요 구성 요소로 이루어집니다. 먼저 **세션 샘플링 및 필터링**을 사용하여 에이전트 상호 작용을 평가할 구체적인 규칙을 구성할 수 있습니다. 백분율 기반 샘플링으로 전체 세션의 일부(예: 10%)를 평가하거나, 더 집중적인 평가를 위한 조건부 필터를 정의할 수 있습니다. 다음으로 새 사용자 지정 evaluator 생성, 기존 사용자 지정 evaluator 사용, 기본 제공 evaluator 선택 등 **여러 평가 방식** 중에서 선택할 수 있습니다. 마지막으로 **모니터링 및 분석** 기능을 통해 대시보드에서 집계 점수를 확인하고, 시간 경과에 따른 품질 추세를 추적하며, 점수가 낮은 세션을 조사하고, 입력부터 출력까지 전체 상호 작용 흐름을 분석할 수 있습니다.

온라인 평가에서는 에이전트 trace가 포함된 CloudWatch log group 또는 AgentCore Runtime endpoint와 같은 특정 데이터 소스를 자동으로 모니터링하도록 시스템을 구성합니다. 서비스는 샘플링 및 필터링 규칙에 따라 수신 trace를 지속적으로 처리하고, 선택한 evaluator를 실시간으로 적용하며, 분석할 수 있도록 상세 결과를 CloudWatch에 출력합니다. 이 평가 유형은 프로덕션 모니터링, 품질 regression 조기 감지, 사용자 상호 작용 패턴 식별, 대규모 환경에서 일관된 에이전트 성능 유지에 특히 유용합니다.

온라인 평가 구성을 생성하고 활성화하면 서비스가 백그라운드에서 지속적으로 실행되며, 세션이 발생할 때 평가하고 에이전트 품질 metric을 계속 확인할 수 있게 해 줍니다. 요구 사항의 변화에 맞춰 언제든지 구성을 일시 중지하거나 수정 또는 삭제하여 평가 전략을 조정할 수 있습니다.

### 에이전트에서 AgentCore Observability trace 생성

AgentCore Observability는 상세한 실행 데이터를 캡처하고 구조화하는 기반으로 [OpenTelemetry (OTEL)](https://opentelemetry.io/) trace를 활용하여 호출 중 에이전트 동작을 포괄적으로 보여줍니다. AgentCore는 [AWS Distro for OpenTelemetry (ADOT)](https://aws-otel.github.io/)를 사용하여 다양한 에이전트 프레임워크에서 여러 유형의 OTEL trace를 계측합니다.

이 튜토리얼의 에이전트처럼 AgentCore Runtime에서 에이전트를 호스팅하면 최소한의 구성만으로 AgentCore Observability 계측이 자동 적용됩니다. `requirements.txt`에 `aws-opentelemetry-distro`를 포함하면 AgentCore Runtime이 OTEL 구성을 자동으로 처리합니다. 에이전트가 AgentCore Runtime에서 실행되지 않는 경우에는 AgentCore Observability에서 사용할 수 있도록 ADOT로 계측해야 합니다. telemetry 데이터를 CloudWatch로 전송하도록 환경 변수를 구성하고 OpenTelemetry 계측을 적용하여 에이전트를 실행해야 합니다.

과정은 다음과 같습니다.

![AgentCore Observability 세션 trace](../../images/observability_traces.png)

세션 trace를 AgentCore Observability에서 사용할 수 있게 되면 AgentCore Evaluations로 에이전트 동작을 평가할 수 있습니다. 온라인 평가에서는 추가 작업이 필요하지 않습니다. 실시간 대시보드에서 에이전트 성능을 모니터링하면 됩니다.

### trace를 사용한 온라인 평가의 작동 방식

온라인 평가에서는 에이전트가 호출되어 AgentCore Observability에 trace를 생성합니다. 이러한 trace는 세션에 매핑되고 해당 로그는 Amazon CloudWatch log group에서 사용할 수 있습니다. 개발자는 특정 에이전트에 대한 온라인 평가 구성을 생성하고, 이 구성에 적용할 샘플링 비율과 evaluator를 정의합니다. 그러면 AgentCore Evaluations가 설정된 샘플링 비율에 따라 생성된 trace를 분석하여 프로덕션의 에이전트를 자동으로 평가합니다. 개발자는 AgentCore Observability 대시보드에서 에이전트의 trace와 평가 점수를 시각화하고 평가 결과에 따라 에이전트를 지속적으로 개선할 수 있습니다.


![온라인 평가 흐름](../../images/online_evaluations.png)

### 이전 튜토리얼의 정보 불러오기

이 튜토리얼에서는 사전 요구 사항 튜토리얼에서 AgentCore Runtime에 배포한 LangGraph 에이전트를 사용합니다. 기본 제공 metric과 `01-creating-custom-metrics` 튜토리얼에서 생성한 `response_quality` metric으로 에이전트를 평가합니다. 에이전트와 evaluator 정보를 불러오겠습니다.

In [ ]:
%store -r launch_result_langgraph
%store -r evaluator_id
try:
    print("Agent Id:", launch_result_langgraph.agent_id)
    print("Agent ARN:", launch_result_langgraph.agent_arn)
except NameError:
    raise Exception(
        """Missing launch results from your LangGraph agent. Please run 00-prereqs before executing this lab"""
    )

try:
    print("Evaluator id:", evaluator_id)
except NameError:
    raise Exception(
        """Missing custom evaluator id. Please run 01-creating-custom-evaluators before executing this lab"""
    )

### AgentCore Evaluations client 초기화

이제 AgentCore Starter toolkit에서 AgentCore Evaluations client를 초기화하겠습니다.

In [ ]:
from bedrock_agentcore_starter_toolkit import Evaluation
import json
from boto3.session import Session
from IPython.display import Markdown, display

In [ ]:
boto_session = Session()
region = boto_session.region_name
print(region)

In [ ]:
eval_client = Evaluation(region=region)

### 온라인 평가 구성 설정

이제 온라인 평가 구성을 설정하겠습니다. 여기서는 데모 목적으로만 에이전트를 사용하므로 생성되는 모든 trace를 평가합니다. 실제 애플리케이션에서는 에이전트 사용량에 맞게 샘플링 비율을 설정해야 합니다.

온디맨드 평가에서 살펴본 다음 5개 metric으로 평가 구성을 생성합니다.
* Builtin.GoalSuccessRate
* Builtin.Correctness
* Builtin.ToolParameterAccuracy
* Builtin.ToolSelectionAccuracy
* 사용자 지정 metric: response_Quality

In [ ]:
response = eval_client.create_online_config(
    agent_id=launch_result_langgraph.agent_id,
    config_name="langgraph_agent_eval",
    sampling_rate=100,
    evaluator_list=[
        "Builtin.GoalSuccessRate",
        "Builtin.Correctness",
        "Builtin.ToolParameterAccuracy",
        "Builtin.ToolSelectionAccuracy",
        evaluator_id,
    ],
    config_description="LangGraph agent online evaluation test",
    auto_create_execution_role=True,
)

### 평가 구성 분석

온라인 평가 구성의 구성 ID를 확인하겠습니다.

In [ ]:
print("Online Evaluation Configuration Id:", response["onlineEvaluationConfigId"])

생성한 구성의 세부 정보를 확인하여 이미 활성화되었는지 확인할 수도 있습니다.

In [ ]:
eval_client.get_online_config(config_id=response["onlineEvaluationConfigId"])

### 평가를 트리거하도록 에이전트 호출

이제 몇 가지 새 query로 에이전트를 호출하여 온라인 평가를 트리거하겠습니다. endpoint를 사용할 수 있게 되면 어떤 interface에서도 호출할 수 있으므로 이번에는 boto3로 에이전트를 호출합니다.

In [ ]:
import boto3

agentcore_client = boto3.client("bedrock-agentcore", region_name=region)


def invoke_agent_runtime(agent_arn, prompt):
    boto3_response = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        qualifier="DEFAULT",
        payload=json.dumps({"prompt": prompt}),
    )
    if "text/event-stream" in boto3_response.get("contentType", ""):
        content = []
        for line in boto3_response["response"].iter_lines(chunk_size=1):
            if line:
                line = line.decode("utf-8")
                if line.startswith("data: "):
                    line = line[6:]
                    print(line)
                    content.append(line)
        display(Markdown("\n".join(content)))
    else:
        try:
            events = []
            for event in boto3_response.get("response", []):
                events.append(event)
        except Exception as e:
            events = [f"Error reading EventStream: {e}"]
        display(Markdown(json.loads(events[0].decode("utf-8"))))
    return boto3_response

In [ ]:
response = invoke_agent_runtime(launch_result_langgraph.agent_arn, "How much is 7+9+10*2?")

In [ ]:
response = invoke_agent_runtime(launch_result_langgraph.agent_arn, "Is it raining?")

In [ ]:
response = invoke_agent_runtime(launch_result_langgraph.agent_arn, "how much is 20% of 300?")

In [ ]:
response = invoke_agent_runtime(launch_result_langgraph.agent_arn, "What can you do?")

In [ ]:
response = invoke_agent_runtime(launch_result_langgraph.agent_arn, "What is the capital of NY State?")

### 온라인 평가 시각화

에이전트와 충분한 상호 작용을 생성한 후 [AgentCore Observability console](https://console.aws.amazon.com/cloudwatch/home#gen-ai-observability/agent-core/agents)에서 온라인 평가 구성에 따른 에이전트 성능을 시각화할 수 있습니다.

현재 평가를 확인하려면 에이전트의 `DEFAULT` endpoint로 이동하세요.

**중요**: 평가 결과가 대시보드에 표시되기까지 시간이 걸릴 수 있습니다. 평가 대시보드가 비어 있다면 몇 분 기다린 후 다시 확인하세요.

결과가 준비되면 에이전트 trace에서 metric을 직접 확인할 수 있습니다.
![온라인 평가 대시보드](../../images/online_evaluations_dashboard.png)

### 축하합니다!

첫 번째 온라인 평가 구성을 생성했습니다. 이제 사용자 지정 metric을 생성하고 AgentCore Evaluations로 에이전트를 온디맨드 및 온라인 방식으로 평가할 수 있습니다.